In [1]:
import sys
from pathlib import Path
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack import Pipeline

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import _META_COLS, MODEL_NAME, mivuls_doc_store

/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
retriever = MilvusEmbeddingRetriever(
    document_store = mivuls_doc_store,
    top_k          = 3
)

## Checks

In [3]:
print("Docs en Milvus:", mivuls_doc_store.count_documents())
assert mivuls_doc_store.count_documents() > 0, "No hay documentos en Milvus"


Docs en Milvus: 218646


In [4]:
text_embedder = SentenceTransformersTextEmbedder(
    model = MODEL_NAME,
)

In [5]:
# --- 3) pipeline -------------------------------------------------
pipe = Pipeline()
pipe.add_component("embedder",  text_embedder)
pipe.add_component("retriever", retriever)
pipe.connect("embedder", "retriever")

# ---------- prueba ----------------------------------------------
question = "How do I chunk text for a RAG pipeline?"
result   = pipe.run({"embedder": {"text": question}})

print(f"\n🔎  {question!r}\n")
for i, doc in enumerate(result["retriever"]["documents"], 1):
    snip = doc.content.replace("\n", " ")[:140] + "…"
    print(f"{i}. score={doc.score:.3f} · {snip}")
    print("   meta →", {k: doc.meta.get(k) for k in _META_COLS if k in doc.meta})


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.28it/s]


🔎  'How do I chunk text for a RAG pipeline?'

1. score=1.091 · paragraphs_for_colbert(INDEX_NAME, PARAGRAPH_FIELD) index_with_colbert(paragraphs, batch_size=BATCH_SIZE) print("Todos los procesos han fina…
   meta → {'message_id': 'aaa23138-948e-492b-976e-7980562b9be0', 'parent_id': '43c56a9c-66f4-4152-8373-2d6f4fd134ae', 'conversation_id': '1ef22661-3cae-537d-ae23-b676598eb276', 'depth': 3, 'order_in_conv': 3, 'role': 'user', 'model_slug': 'gpt-4o', 'conversation_ttl': 'Indexación documentos legales', 'created_at': '2024-12-08T19:18:02.774674'}
2. score=1.192 · ollow](  <p align="center"><img width=500 alt="The RAGatouille logo, it's a cheerful rat on his laptop (branded with a slightly eaten piece …
   meta → {'message_id': 'aaa23138-948e-492b-976e-7980562b9be0', 'parent_id': '43c56a9c-66f4-4152-8373-2d6f4fd134ae', 'conversation_id': '1ef22661-3cae-537d-ae23-b676598eb276', 'depth': 3, 'order_in_conv': 3, 'role': 'user', 'model_slug': 'gpt-4o', 'conversation_ttl': 'Indexación document

In [6]:
result

{'retriever': {'documents': [Document(id=3125251574c1dd6bd1f8f17fa76d2f02d059083a5cafea58fea2f942b367569f, content: 'paragraphs_for_colbert(INDEX_NAME, PARAGRAPH_FIELD)
   index_with_colbert(paragraphs, batch_size=BATCH_...', meta: {'message_id': 'aaa23138-948e-492b-976e-7980562b9be0', 'parent_id': '43c56a9c-66f4-4152-8373-2d6f4fd134ae', 'conversation_id': '1ef22661-3cae-537d-ae23-b676598eb276', 'depth': 3, 'order_in_conv': 3, 'role': 'user', 'model_slug': 'gpt-4o', 'conversation_ttl': 'Indexación documentos legales', 'created_at': '2024-12-08T19:18:02.774674'}, score: 1.0910778045654297, embedding: vector of size 384),
   Document(id=a548fd681ba9a9d8e02c1173f6e7c5c6956e736ee941dbbb1579d7dcf1ed07f3, content: 'ollow](
   
   <p align="center"><img width=500 alt="The RAGatouille logo, it's a cheerful rat on his lapt...', meta: {'message_id': 'aaa23138-948e-492b-976e-7980562b9be0', 'parent_id': '43c56a9c-66f4-4152-8373-2d6f4fd134ae', 'conversation_id': '1ef22661-3cae-537d-ae23-b676598eb27